# Virtuális próbafülke bemutató (Virtual try-on)

Egy személy fotója és egy ruhadarab. A `virtual-try-on-001` modell ebből új képet készít: a személyt a kiválasztott ruhában mutatja.

A modell nem szöveggel válaszol, hanem egy képet generál. (A hívás a GCP predict végpontjára történik)

## Mi az a Jupyter notebook?

Ez a fájl egy Jupyter notebook. A kiterjesztése `.ipynb`. A magyarázat és a futtatható kód ugyanabban a dokumentumban van, egymás alatti cellákban.

Egy cella vagy szöveget tartalmaz, vagy egy parancsot. A parancsot a cella bal oldalán lévő lejátszás gombbal futtatod. Az eredmény a cella alatt jelenik meg: szöveg vagy kép.

A Google Colab egy Jupyter notebook a böngészőben. Külön programot nem kell hozzá telepíteni a saját gépre.

## Hogyan futtasd

1. Nyisd meg a [Colabot](https://colab.research.google.com) azzal a Google-fiókkal, amelyiknek hozzáférése van a GCP projekthez.
2. **Fájl > Notebook feltöltése**, és válaszd ezt a fájlt.
3. Futtasd a cellákat egyesével, fentről lefelé. A szükséges paraméter még futtatás előtt állítsd be, ott ahol az megjelenik.
4. Amikor a Google bejelentkezési ablak felugrik, ugyanazt a fiókot válaszd.


## 1. Szükséges komponensek telepítése

<p style="margin:14px 0 4px 0"><b style="color:#1A73E8">Mit csinál</b></p>
<div style="background-color:#E8F0FE;border-left:4px solid #1A73E8;color:#202124;padding:10px 14px;border-radius:6px">
<p>Két csomag kell ehhez a megoldáshoz.</p>
<ul>
<li>A <code>google-auth</code> kéri le a belépési tokent a GCP-hez.</li>
<li>A <code>requests</code> tölti le a képeket, és küldi el a predict hívást.</li>
</ul>
</div>

<p style="margin:14px 0 4px 0"><b style="color:#E37400">Hogyan használd</b></p>
<div style="background-color:#FEF7E0;border-left:4px solid #E37400;color:#202124;padding:10px 14px;border-radius:6px">
<p>Futtasd a cellát.</p>
</div>

<p style="margin:14px 0 4px 0"><b style="color:#188038">Mi történik</b></p>
<div style="background-color:#E6F4EA;border-left:4px solid #188038;color:#202124;padding:10px 14px;border-radius:6px">
<p>A parancs kiírja, mit telepít. Ha azt látod, hogy a csomag már fent van, mehetsz tovább.</p>
</div>


In [ ]:
%pip install google-auth requests


## 2. Bejelentkezés a Google Cloud Projektbe

<p style="margin:14px 0 4px 0"><b style="color:#1A73E8">Mit csinál</b></p>
<div style="background-color:#E8F0FE;border-left:4px solid #1A73E8;color:#202124;padding:10px 14px;border-radius:6px">
<p>A <code>PROJECT_ID</code> annak a GCP projektnek az azonosítója, amelyben a modell fut. A hívás ehhez a projekthez tartozik.</p>
</div>

<p style="margin:14px 0 4px 0"><b style="color:#E37400">Hogyan használd</b></p>
<div style="background-color:#FEF7E0;border-left:4px solid #E37400;color:#202124;padding:10px 14px;border-radius:6px">
<p>Írd be az azonosítót, majd futtasd a cellát. A felugró ablakban azt a fiókot válaszd, amelyiknek ehhez a projekthez van hozzáférése.</p>
</div>

<p style="margin:14px 0 4px 0"><b style="color:#188038">Mi történik</b></p>
<div style="background-color:#E6F4EA;border-left:4px solid #188038;color:#202124;padding:10px 14px;border-radius:6px">
<p>Sikeres futás után ez a sor jelenik meg: <code>Bejelentkezés kész. Projekt: ...</code></p>
</div>


In [ ]:
import base64
import json
import os
import sys

import google.auth
import google.auth.transport.requests
import requests
from IPython.display import Image, display

PROJECT_ID = ""  # @param {type: "string"}

if not PROJECT_ID:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT_ID") or ""
if not PROJECT_ID:
    raise RuntimeError('Írd be a projektazonosítót: PROJECT_ID = "a-projekt-id"')

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user(project_id=PROJECT_ID, clear_output=False)

print(f"Bejelentkezés kész. Projekt: {PROJECT_ID}")


## 3. A személy képének kiválasztása

<p style="margin:14px 0 4px 0"><b style="color:#1A73E8">Mit csinál</b></p>
<div style="background-color:#E8F0FE;border-left:4px solid #1A73E8;color:#202124;padding:10px 14px;border-radius:6px">
<p>Ez a fotó, akire a ruha kerül. A legördülő menüben négy minta van: Férfi 1, Férfi 2, Nő 1, Nő 2.</p>
</div>

<p style="margin:14px 0 4px 0"><b style="color:#E37400">Hogyan használd</b></p>
<div style="background-color:#FEF7E0;border-left:4px solid #E37400;color:#202124;padding:10px 14px;border-radius:6px">
<p>Ha saját képet szeretnél használni, akkor még futtatás előtt válaszd az <b>Egyedi URL</b> sort, és illeszd be a kép <code>https://</code> linkjét a második mezőbe. A link egyenesen a jpg vagy png fájlra mutasson.</p>
</div>

<p style="margin:14px 0 4px 0"><b style="color:#188038">Mi történik</b></p>
<div style="background-color:#E6F4EA;border-left:4px solid #188038;color:#202124;padding:10px 14px;border-radius:6px">
<p>Futtasd a cellát. A parancs kiírja a választott nevet, a linket és a letöltés eredményét, majd megjeleníti a képet.</p>
</div>


In [ ]:
PERSON_IMAGE = "Férfi 1"  # @param ["Férfi 1", "Férfi 2", "Nő 1", "Nő 2", "Egyedi URL"]
PERSON_IMAGE_CUSTOM_URL = ""  # @param {type: "string"}
PERSON_IMAGES = {
    "Férfi 1": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/modell/ferfi_1.jpg",
    "Férfi 2": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/modell/ferfi_2.jpg",
    "Nő 1": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/modell/no_1.jpg",
    "Nő 2": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/modell/no_2.jpg",
}
if PERSON_IMAGE == "Egyedi URL":
    PERSON_IMAGE_URL = PERSON_IMAGE_CUSTOM_URL.strip()
    if not PERSON_IMAGE_URL.startswith("https://"):
        raise ValueError("Egyedi képhez https URL kell a PERSON_IMAGE_CUSTOM_URL mezőben.")
else:
    PERSON_IMAGE_URL = PERSON_IMAGES[PERSON_IMAGE]

print(f"Kiválasztott modell: {PERSON_IMAGE}")
print(f"Link: {PERSON_IMAGE_URL}")
image_response = requests.get(PERSON_IMAGE_URL, timeout=60)
print(f"Letöltés kész. HTTP {image_response.status_code}, {len(image_response.content)} byte.")
image_response.raise_for_status()
person_bytes = image_response.content
display(Image(data=person_bytes, width=450))


## 4. A ruhadarab kiválasztása

<p style="margin:14px 0 4px 0"><b style="color:#1A73E8">Mit csinál</b></p>
<div style="background-color:#E8F0FE;border-left:4px solid #1A73E8;color:#202124;padding:10px 14px;border-radius:6px">
<p>Eszerre csak egy ruhadarabot adhatunk a személyre. A listában férfi és női felsők, nadrágok, szoknyák és lábbelik vannak.</p>
</div>

<p style="margin:14px 0 4px 0"><b style="color:#E37400">Hogyan használd</b></p>
<div style="background-color:#FEF7E0;border-left:4px solid #E37400;color:#202124;padding:10px 14px;border-radius:6px">
<p>Saját kép használatához még futtatás előtt válaszd az <b>Egyedi URL</b> sort, és illeszd be a linkjét. A képen a ruhadarab legyen, egyszerű, világos háttér előtt.</p>
</div>

<p style="margin:14px 0 4px 0"><b style="color:#188038">Mi történik</b></p>
<div style="background-color:#E6F4EA;border-left:4px solid #188038;color:#202124;padding:10px 14px;border-radius:6px">
<p>Futtasd a cellát. Alatta megjelenik a kiválasztott darab. Ezt a képet küldi el a következő lépés ruhának.</p>
</div>


In [ ]:
GARMENT_IMAGE = "Férfi ing"  # @param ["Férfi cipő", "Férfi ing", "Férfi mellény", "Férfi nadrág", "Férfi papucs 1", "Férfi papucs 2", "Férfi póló", "Férfi rövidnadrág", "Női cipő 1", "Női cipő 2", "Női hosszú ujjú felső", "Női mellény", "Női nadrág", "Női ruha", "Női szandál", "Női szoknya 1", "Női szoknya 2", "Női top", "Egyedi URL"]
GARMENT_IMAGE_CUSTOM_URL = ""  # @param {type: "string"}
GARMENT_IMAGES = {
    "Férfi cipő": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/f_cipo_1.jpg",
    "Férfi ing": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/f_ing.jpg",
    "Férfi mellény": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/f_melleny.jpg",
    "Férfi nadrág": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/f_nadrag.jpg",
    "Férfi papucs 1": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/f_papucs_1.jpg",
    "Férfi papucs 2": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/f_papucs_2.jpg",
    "Férfi póló": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/f_polo.jpg.jpg",
    "Férfi rövidnadrág": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/f_rovid_nadrag.jpg",
    "Női cipő 1": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/n_cipo_1.jpg",
    "Női cipő 2": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/n_cipo_2.jpg",
    "Női hosszú ujjú felső": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/n_hosszaujju-felso.jpg",
    "Női mellény": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/n_melleny.jpg",
    "Női nadrág": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/n_nadrag.jpg",
    "Női ruha": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/n_ruha.jpg",
    "Női szandál": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/n_szandal.jpg",
    "Női szoknya 1": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/n_szoknya_1.jpg",
    "Női szoknya 2": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/n_szoknya_2.jpg",
    "Női top": "https://raw.githubusercontent.com/cloudsteak/trn-gcp-ai-virtual-tryon/main/images/ruha/n_top.jpg",
}
if GARMENT_IMAGE == "Egyedi URL":
    GARMENT_IMAGE_URL = GARMENT_IMAGE_CUSTOM_URL.strip()
    if not GARMENT_IMAGE_URL.startswith("https://"):
        raise ValueError("Egyedi képhez https URL kell a GARMENT_IMAGE_CUSTOM_URL mezőben.")
else:
    GARMENT_IMAGE_URL = GARMENT_IMAGES[GARMENT_IMAGE]

print(f"Kiválasztott ruhadarab: {GARMENT_IMAGE}")
print(f"Link: {GARMENT_IMAGE_URL}")
image_response = requests.get(GARMENT_IMAGE_URL, timeout=60)
print(f"Letöltés kész. HTTP {image_response.status_code}, {len(image_response.content)} byte.")
image_response.raise_for_status()
garment_bytes = image_response.content
display(Image(data=garment_bytes, width=450))


## 5. Az öltöztetés - A kép generálása

<p style="margin:14px 0 4px 0"><b style="color:#1A73E8">Mit csinál</b></p>
<div style="background-color:#E8F0FE;border-left:4px solid #1A73E8;color:#202124;padding:10px 14px;border-radius:6px">
<p>A cella a <code>virtual-try-on-001</code> predict végpontját hívja. A két előző kép base64 szövegként megy a kérésbe. A <code>baseSteps</code> értéke 32, ez a modell alapértéke. A kiírásban a kép helyén helyőrző látszik, mert a nyers base64 túl hosszú lenne.</p>
<pre style="background:#fff;color:#202124;padding:8px 12px;border-radius:6px;overflow:auto">POST https://REGION-aiplatform.googleapis.com/v1/projects/PROJECT_ID/locations/REGION/publishers/google/models/virtual-try-on-001:predict</pre>
</div>

<p style="margin:14px 0 4px 0"><b style="color:#E37400">Hogyan használd</b></p>
<div style="background-color:#FEF7E0;border-left:4px solid #E37400;color:#202124;padding:10px 14px;border-radius:6px">
<p>A régiót még futtatás előtt ellenőrizd. Az alapérték <code>europe-west1</code>. Ezt használd, hacsak a modellt nem másik régióban kapcsoltad be.</p>
</div>

<p style="margin:14px 0 4px 0"><b style="color:#188038">Mi történik</b></p>
<div style="background-color:#E6F4EA;border-left:4px solid #188038;color:#202124;padding:10px 14px;border-radius:6px">
<p>A generálás körülbelül egy perc. A cella kiírja a végpontot, a kérés vázát és a HTTP választ, majd megmutatja az elkészült képet.</p>
</div>


In [ ]:
LOCATION = "europe-west1"  # @param {type: "string"}
MODEL_NAME = "virtual-try-on-001"
BASE_STEPS = 32

try:
    credentials, _ = google.auth.default()
    credentials.refresh(google.auth.transport.requests.Request())
except google.auth.exceptions.DefaultCredentialsError as exc:
    raise RuntimeError(
        "Nincs GCP bejelentkezés. Futtasd újra a projekt cellát, és a GCP-s Google-fiókot válaszd."
    ) from exc

b64_person_image = base64.b64encode(person_bytes).decode("utf-8")
b64_product_image = base64.b64encode(garment_bytes).decode("utf-8")
payload = {
    "instances": [
        {
            "personImage": {
                "image": {"bytesBase64Encoded": b64_person_image}
            },
            "productImages": [
                {"image": {"bytesBase64Encoded": b64_product_image}}
            ],
        }
    ],
    "parameters": {"baseSteps": BASE_STEPS},
}
preview = {
    "instances": [
        {
            "personImage": {"image": {"bytesBase64Encoded": "${B64_PERSON_IMAGE}"}},
            "productImages": [{"image": {"bytesBase64Encoded": "${B64_PRODUCT_IMAGE}"}}],
        }
    ],
    "parameters": {"baseSteps": BASE_STEPS},
}
url = (
    f"https://{LOCATION}-aiplatform.googleapis.com/v1/projects/{PROJECT_ID}"
    f"/locations/{LOCATION}/publishers/google/models/{MODEL_NAME}:predict"
)
print(f"Végpont: {url}")
print("Elküldött kérés, a képek helyén helyőrzővel:")
print(json.dumps(preview, indent=2, ensure_ascii=False))

response = requests.post(
    url,
    data=json.dumps(payload).encode("utf-8"),
    headers={
        "Authorization": f"Bearer {credentials.token}",
        "Content-Type": "application/json; charset=utf-8",
        "x-goog-user-project": PROJECT_ID,
    },
    timeout=180,
)
print(f"Válasz: HTTP {response.status_code}")
if not response.ok:
    print(response.text[:500])
    raise RuntimeError(f"A modellhívás sikertelen ({response.status_code}).")

prediction = response.json()["predictions"][0]
result_bytes = base64.b64decode(prediction["bytesBase64Encoded"])
print(f"Generált kép: {prediction.get('mimeType', 'image/png')}, {len(result_bytes)} byte.")
display(Image(data=result_bytes, width=450))
